# FedKG-Maint — Phase 3 Privacy/Utility: Local-Normalization Compliance Run v1

This is a **separate validation-selected experiment** prepared after the Azure-only requirements audit. The existing Phase 3 Privacy Utility Fix v3 used `policy="global"` for train, validation, and test. The project specification requires normalization statistics to be computed locally inside each factory, so this notebook reruns the same compact DP-FL experiment with `policy="local"`.

## Protocol
- Keep the same 7 predeclared past-only features, 8-parameter logistic model, four training factories, held-out `factory_5`, train/validation splits, three rounds, three seeds, clipping grid, noise grid, and epsilon cap.
- Use the per-client scalers saved by Phase 1 v3. Factories 1–4 use scalers fitted on their own training rows. `factory_5` uses only its earlier feature-only unlabeled calibration rows; this is unsupervised site adaptation, not zero-shot normalization.
- Use `policy="local"` for training, validation, and the final test load. The selected model/configuration is locked using validation only before test data are opened.
- Write all outputs to `phase3_federated_privacy_localnorm_v1/`; do not overwrite the prior global-policy run.
- Compare DP-FedAvg with matched clipped/no-noise and ordinary FedAvg compact models, plus a pooled central compact reference using the same local-scaled feature matrix.

**Important:** this notebook has been prepared by changing the preprocessing policy, but it has not been run on the user's Google Drive data. Its cleared output cells are intentional. Do not use the earlier global-policy results as outputs for this local-normalization run. `test_site_full` was used in earlier project experiments; this remains a benchmark analysis, not a new untouched external test.

**Privacy scope:** the accountant covers only the clipped/noised client-update channel under trusted-server, fixed-denominator, full-participation, add/remove-one-client assumptions and no subsampling amplification. The preprocessing notebook still simulates the factories in a centralized process; this is not proof of physically enforced raw-data isolation, and the guarantee is not end-to-end DP.


## 1. Environment and prerequisite checks

This run reads the same `prep_v3/` data but uses the `local` scaler per client and writes into a new output directory. The previous global-policy run remains unchanged.


In [ ]:
import os, sys, json, math, time, random, warnings, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore", category=FutureWarning)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

PREP_DIR = PROJECT_DIR / "prep_v3"
P2_DIR = PROJECT_DIR / "phase2_baselines_prep_v3"
P3_DIR = PROJECT_DIR / "phase3_federated_prep_v3_privacy_v2"
OUT_DIR = PROJECT_DIR / "phase3_federated_privacy_localnorm_v1"
for sub in ["models", "predictions", "results", "reports", "logs"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

required = [PREP_DIR / "fedkg_data.py",
            PREP_DIR / "configs" / "feature_groups.json",
            P2_DIR / "fedkg_eval.py",
            P3_DIR / "results" / "federated_results.csv"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run Phase 1 v3, Phase 2 v3 and Phase 3 v2 first:\n" + "\n".join(missing))

sys.path.insert(0, str(PREP_DIR)); sys.path.insert(0, str(P2_DIR))
sys.modules.pop("fedkg_data", None); sys.modules.pop("fedkg_eval", None)
importlib.invalidate_caches()
import fedkg_data
from fedkg_eval import SetEval, draw_weights, ci
FEATURE_CFG = json.loads((PREP_DIR / "configs" / "feature_groups.json").read_text())
P3_BASE = pd.read_csv(P3_DIR / "results" / "federated_results.csv")

NORMALIZATION_POLICY = "local"  # required per-factory normalization protocol
SEEDS = [42, 43, 44]
CLIENTS = [f"factory_{i}" for i in range(1, 5)]
HELD_OUT = "factory_5"
ROUNDS = 3
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
POS_WEIGHT = 20.0
CLIP_CANDIDATES = [0.02, 0.05, 0.1, 0.2, 0.5]
LOCAL_EPOCH_CANDIDATES = [1, 3]
NOISE_CANDIDATES = [1.0, 1.25, 1.5, 2.0]
EPSILON_CAP = 10.0
DELTA = 1e-5
BOOTSTRAP_B = 1000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_CUTOFF = pd.Timestamp(FEATURE_CFG["boundaries"]["train_cutoff"])

# Predeclared compact feature group: 4 current sensors + 2 error-history features + maintenance recency.
FEATURE_NAMES = (FEATURE_CFG["groups"]["sensor_raw"] +
                 FEATURE_CFG["groups"]["error_recency"] +
                 FEATURE_CFG["groups"]["maintenance_recency"])
FULL_FEATURES = FEATURE_CFG["feature_sets"]["full"]
FEATURE_IDXS = [FULL_FEATURES.index(name) for name in FEATURE_NAMES]
assert len(FEATURE_NAMES) == 7 and len(set(FEATURE_NAMES)) == 7
assert set(CLIENTS).isdisjoint({HELD_OUT})
assert set(fedkg_data.SPLITS) >= {"train", "validation", "test_site_full"}
assert P3_BASE["eval_set"].eq("test_site_full").any()
print("Device:", DEVICE)
print("Project:", PROJECT_DIR)
print("Output:", OUT_DIR)
print("Compact features (7):", FEATURE_NAMES)
print("Model parameter count:", len(FEATURE_NAMES) + 1)
print("Clients:", CLIENTS, "| held out:", HELD_OUT)
print("Rounds:", ROUNDS, "| local epochs:", LOCAL_EPOCH_CANDIDATES, "| clipping candidates:", CLIP_CANDIDATES, "| noise candidates:", NOISE_CANDIDATES)


## 2. Model, utility functions, and accounting

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def make_model(n_features, seed):
    # Zero initialization avoids making the tiny linear model's start depend on seed.
    # Seed still controls minibatch order and Gaussian DP noise.
    torch.manual_seed(int(seed))
    model = nn.Linear(n_features, 1, bias=True).to(DEVICE)
    with torch.no_grad():
        model.weight.zero_()
        model.bias.zero_()
    return model

def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def update_vector(model):
    return parameters_to_vector([p.detach() for p in model.parameters()]).detach().clone()

def load_vector(model, vector):
    vector_to_parameters(vector, model.parameters())

def data_tensor(X, y=None):
    xt = torch.as_tensor(np.asarray(X, dtype=np.float32), dtype=torch.float32, device=DEVICE)
    if y is None: return xt
    yt = torch.as_tensor(np.asarray(y, dtype=np.float32), dtype=torch.float32, device=DEVICE).reshape(-1, 1)
    return xt, yt

def train_one_epoch(model, X_t, y_t, row_indices, optimizer):
    model.train()
    order = row_indices[torch.randperm(len(row_indices), device=DEVICE)]
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    total_loss, seen = 0.0, 0
    for start in range(0, len(order), BATCH_SIZE):
        idx = order[start:start+BATCH_SIZE]
        xb = X_t.index_select(0, idx); yb = y_t.index_select(0, idx)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(xb), yb)
        loss.backward(); optimizer.step()
        n = len(idx); total_loss += float(loss.detach().cpu()) * n; seen += n
    return total_loss / max(seen, 1)

@torch.no_grad()
def predict_scores(model, X, batch_size=32768):
    model.eval()
    xt = torch.as_tensor(np.asarray(X, dtype=np.float32), dtype=torch.float32, device=DEVICE)
    out=[]
    for start in range(0, len(xt), batch_size):
        out.append(torch.sigmoid(model(xt[start:start+batch_size]).reshape(-1)).detach().cpu().numpy())
    return np.concatenate(out) if out else np.empty(0, dtype=np.float64)

def privacy_epsilon(noise_multiplier, steps=ROUNDS, delta=DELTA):
    """RDP bound for full participation; no subsampling amplification."""
    if noise_multiplier <= 0: return float("inf"), None
    orders = np.array([1.25, 1.5, 2, 3, 4, 5, 8, 10, 16, 32, 64, 128, 256, 512], dtype=float)
    rdp = steps * orders / (2.0 * noise_multiplier**2)
    eps = rdp + math.log(1.0/delta)/(orders-1.0)
    i = int(np.argmin(eps))
    return float(eps[i]), float(orders[i])

def clip_update(delta, clip_norm):
    norm = torch.linalg.vector_norm(delta)
    factor = torch.clamp(torch.tensor(float(clip_norm), device=delta.device)/(norm+1e-12), max=1.0)
    return delta * factor

def validation_metrics(val_eval, scores, threshold=None):
    packed = val_eval.pack(scores)
    if threshold is None:
        threshold = float(val_eval.best_f1_threshold(packed))
    metrics = val_eval.metrics(packed, None, threshold)
    sklearn_ap = float(average_precision_score(val_eval.y, scores))
    # SetEval's AP should match row-level sklearn AP; fail loudly if evaluator semantics drift.
    if not np.isclose(float(metrics["ap"]), sklearn_ap, atol=1e-8, rtol=1e-7):
        raise AssertionError(f"Validation AP mismatch: SetEval={metrics['ap']}, sklearn={sklearn_ap}")
    return float(metrics["ap"]), sklearn_ap, float(threshold)

acct = pd.DataFrame([{"noise_multiplier": s, "epsilon": privacy_epsilon(s)[0], "delta": DELTA,
                      "rounds": ROUNDS, "epsilon_cap_10": privacy_epsilon(s)[0] <= EPSILON_CAP}
                     for s in NOISE_CANDIDATES])
print(acct.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
assert acct["epsilon_cap_10"].all(), "Every declared noise candidate must meet the predeclared epsilon cap."

## 3. Load training and validation only

Train and validation are loaded with `NORMALIZATION_POLICY="local"`. Each training factory is transformed using its own scaler; factory_5 uses its disjoint feature-only calibration scaler when test data are later loaded.


In [ ]:
Xtr_all, ytr, mtr = fedkg_data.load_split("train", policy=NORMALIZATION_POLICY, feature_set="full")
Xv_all, yv, mv = fedkg_data.load_split("validation", policy=NORMALIZATION_POLICY, feature_set="full")
Xtr = Xtr_all[:, FEATURE_IDXS].copy(); Xv = Xv_all[:, FEATURE_IDXS].copy()
assert NORMALIZATION_POLICY == "local", "This compliance run must use per-factory local normalization."
assert set(mtr["client_id"].unique()) == set(CLIENTS)
assert HELD_OUT not in set(mtr["client_id"].unique())
assert HELD_OUT not in set(mv["client_id"].unique())
assert np.isfinite(Xtr).all() and np.isfinite(Xv).all()

# Validation failure events only; no held-out test events are used in this selection stage.
val_machines = set(mv["machineID"].unique())
val_chunks = []
for chunk in pd.read_csv(PROJECT_DIR / "data" / "raw" / "PdM_failures.csv",
                         usecols=["machineID", "datetime"], chunksize=100_000):
    chunk = chunk[chunk["machineID"].isin(val_machines)].copy()
    if len(chunk): val_chunks.append(chunk)
val_failures = pd.concat(val_chunks, ignore_index=True) if val_chunks else pd.DataFrame(columns=["machineID", "datetime"])
val_failures["datetime"] = pd.to_datetime(val_failures["datetime"])
val_failures = val_failures.drop_duplicates().sort_values(["machineID", "datetime"])
VAL_EVAL = SetEval(mv.reset_index(drop=True), yv, val_failures)
CLIENT_IDX = {c: torch.as_tensor(np.flatnonzero(mtr["client_id"].to_numpy() == c), dtype=torch.long, device=DEVICE)
              for c in CLIENTS}
SIZES = {c: len(CLIENT_IDX[c]) for c in CLIENTS}
assert len(set(SIZES.values())) == 1 and all(v > 0 for v in SIZES.values())
print("Train:", Xtr.shape, "Validation:", Xv.shape, "Client row counts:", SIZES)
print("Validation positive rate:", float(np.mean(yv)))


## 4. Train compact-linear FL models and perform a validation-only sweep

The selection criterion is row-level AP, cross-checked with `SetEval`. Clipping norm, local epochs, and noise multiplier are selected using validation only. Test data are not loaded in this section.


In [ ]:
def fit_federated_linear(seed, X_train, y_train, meta_train, X_val, y_val, val_eval,
                         method, clip_norm=None, noise_multiplier=0.0, local_epochs=1):
    """Methods: FedAvg, ClippedFedAvg, DP-FedAvg. Checkpoint chosen by validation AP."""
    seed_everything(seed)
    X_t, y_t = data_tensor(X_train, y_train)
    client_idx = {c: torch.as_tensor(np.flatnonzero(meta_train["client_id"].to_numpy() == c),
                                     dtype=torch.long, device=DEVICE) for c in CLIENTS}
    sizes = {c: len(client_idx[c]) for c in CLIENTS}
    if len(set(sizes.values())) != 1 or any(v == 0 for v in sizes.values()):
        raise ValueError(f"Equal non-empty clients required: {sizes}")
    model = make_model(X_train.shape[1], seed)
    history=[]
    best={"ap":-np.inf, "round":None, "state":None, "threshold":None, "row_ap":None}
    for rnd in range(1, ROUNDS+1):
        base = update_vector(model)
        deltas=[]
        for client in CLIENTS:
            local = make_model(X_train.shape[1], seed + rnd*100 + int(client.split("_")[-1]))
            local.load_state_dict(model.state_dict())
            opt = torch.optim.Adam(local.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            # More local epochs improve local optimization without increasing communication rounds.
            # Only the final clipped client update is released in each round.
            epoch_losses=[]
            for _ in range(int(local_epochs)):
                epoch_losses.append(train_one_epoch(local, X_t, y_t, client_idx[client], opt))
            delta = update_vector(local) - base
            if method in ("ClippedFedAvg", "DP-FedAvg"):
                delta = clip_update(delta, clip_norm)
            deltas.append(delta)
            del local, opt
        mean_delta = torch.stack(deltas, dim=0).mean(dim=0)
        if method == "DP-FedAvg":
            # Add/remove client adjacency, fixed public denominator, sensitivity C / N_clients.
            noise_std = float(noise_multiplier) * float(clip_norm) / len(CLIENTS)
            mean_delta = mean_delta + torch.normal(0.0, noise_std, size=mean_delta.shape, device=DEVICE)
        load_vector(model, base + mean_delta)
        scores = predict_scores(model, X_val)
        ap, row_ap, threshold = validation_metrics(val_eval, scores)
        history.append({"seed": seed, "method": method, "round": rnd,
                        "validation_AP": ap, "validation_row_AP": row_ap,
                        "clip_norm": clip_norm, "noise_multiplier": noise_multiplier,
                        "local_epochs": int(local_epochs)})
        if ap > best["ap"]:
            best={"ap":ap, "row_ap":row_ap, "round":rnd, "state":cpu_state(model), "threshold":threshold}
    return {"seed":seed, "method":method, "state":best["state"], "validation_ap":best["ap"],
            "validation_row_ap":best["row_ap"], "selected_round":int(best["round"]),
            "threshold":float(best["threshold"]), "clip_norm":clip_norm,
            "noise_multiplier":float(noise_multiplier), "local_epochs":int(local_epochs),
            "epsilon":privacy_epsilon(noise_multiplier)[0] if method == "DP-FedAvg" else np.nan,
            "n_features":X_train.shape[1], "history":history}

def fit_central_linear(seed, X_train, y_train, X_val, y_val, val_eval, total_epochs):
    seed_everything(seed)
    X_t, y_t = data_tensor(X_train, y_train)
    idx = torch.arange(len(X_train), dtype=torch.long, device=DEVICE)
    model=make_model(X_train.shape[1], seed)
    opt=torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best={"ap":-np.inf, "round":None, "state":None, "threshold":None, "row_ap":None}; history=[]
    for epoch in range(1, int(total_epochs)+1):
        train_one_epoch(model, X_t, y_t, idx, opt)
        scores=predict_scores(model,X_val)
        ap,row_ap,threshold=validation_metrics(val_eval,scores)
        history.append({"seed":seed,"method":"CentralCompactLinear","round":epoch,
                        "validation_AP":ap,"validation_row_AP":row_ap,"local_epochs":np.nan})
        if ap > best["ap"]:
            best={"ap":ap,"row_ap":row_ap,"round":epoch,"state":cpu_state(model),"threshold":threshold}
    return {"seed":seed,"method":"CentralCompactLinear","state":best["state"],
            "validation_ap":best["ap"],"validation_row_ap":best["row_ap"],
            "selected_round":int(best["round"]),"threshold":float(best["threshold"]),
            "local_epochs":np.nan,"n_features":X_train.shape[1],"history":history}

SWEEP=[]; SWEEP_HISTORY=[]; DP_CANDIDATES=[]
start=time.time()
for seed in SEEDS:
    for local_epochs in LOCAL_EPOCH_CANDIDATES:
        for clip in CLIP_CANDIDATES:
            for sigma in NOISE_CANDIDATES:
                art=fit_federated_linear(seed, Xtr, ytr, mtr, Xv, yv, VAL_EVAL,
                                         "DP-FedAvg", clip_norm=clip, noise_multiplier=sigma,
                                         local_epochs=local_epochs)
                art["key"] = f"DP-FedAvg-CompactLinear|seed{seed}|ep{local_epochs}|clip{clip}|sigma{sigma}"
                DP_CANDIDATES.append(art)
                SWEEP_HISTORY.extend(art.pop("history"))
                SWEEP.append({"seed":seed,"local_epochs":local_epochs,"clip_norm":clip,
                              "noise_multiplier":sigma,"epsilon":art["epsilon"],
                              "validation_AP":art["validation_ap"],
                              "validation_row_AP":art["validation_row_ap"],
                              "selected_round":art["selected_round"],"n_parameters":len(FEATURE_NAMES)+1})
                print(f"seed={seed} epochs={local_epochs} clip={clip:.3g} sigma={sigma:.2f} eps={art['epsilon']:.3f} val_AP={art['validation_ap']:.4f} best_round={art['selected_round']}")

SWEEP_DF=pd.DataFrame(SWEEP)
SWEEP_DF.to_csv(OUT_DIR/"results"/"compact_dp_sweep_validation_only.csv", index=False)
pd.DataFrame(SWEEP_HISTORY).to_csv(OUT_DIR/"logs"/"compact_dp_learning_curves.csv", index=False)
SWEEP_SUMMARY=(SWEEP_DF.groupby(["local_epochs","clip_norm","noise_multiplier"], as_index=False)
               .agg(epsilon=("epsilon","first"), mean_validation_AP=("validation_AP","mean"),
                    sd_validation_AP=("validation_AP","std"), seeds=("seed","nunique"),
                    mean_selected_round=("selected_round","mean")))
SWEEP_SUMMARY["eligible"] = SWEEP_SUMMARY["epsilon"] <= EPSILON_CAP
SWEEP_SUMMARY.to_csv(OUT_DIR/"results"/"compact_dp_sweep_summary_validation_only.csv", index=False)
eligible=SWEEP_SUMMARY[SWEEP_SUMMARY["eligible"]].copy()
if eligible.empty: raise RuntimeError("No compact DP configuration meets the predeclared epsilon cap.")
# One shared configuration selected by mean validation AP only. Tie-break toward stronger noise,
# then fewer local epochs, then the smaller clipping norm.
chosen=eligible.sort_values(["mean_validation_AP","noise_multiplier","local_epochs","clip_norm"],
                            ascending=[False,False,True,True]).iloc[0]
selected_local_epochs=int(chosen["local_epochs"])
selected_clip=float(chosen["clip_norm"]); selected_sigma=float(chosen["noise_multiplier"])
SELECTED={a["seed"]:a for a in DP_CANDIDATES
          if a["local_epochs"]==selected_local_epochs
          and np.isclose(a["clip_norm"],selected_clip)
          and np.isclose(a["noise_multiplier"],selected_sigma)}
assert set(SELECTED)==set(SEEDS)

# Matched non-private references use the selected local-epoch count and same data/features.
BASELINES=[]; BASE_HISTORY=[]
for seed in SEEDS:
    fa=fit_federated_linear(seed,Xtr,ytr,mtr,Xv,yv,VAL_EVAL,"FedAvg",local_epochs=selected_local_epochs)
    cf=fit_federated_linear(seed,Xtr,ytr,mtr,Xv,yv,VAL_EVAL,"ClippedFedAvg",
                            clip_norm=selected_clip,local_epochs=selected_local_epochs)
    cen=fit_central_linear(seed,Xtr,ytr,Xv,yv,VAL_EVAL,total_epochs=ROUNDS*selected_local_epochs)
    for a in [fa,cf,cen]:
        a["key"]=f"{a['method']}|CompactLinear|seed{seed}"
        BASE_HISTORY.extend(a.pop("history"))
        BASELINES.append(a)
        torch.save({"state":a["state"],"method":a["method"],"seed":seed,"feature_names":FEATURE_NAMES,
                    "selected_round":a["selected_round"],"validation_ap":a["validation_ap"],
                    "threshold":a["threshold"],"local_epochs":a.get("local_epochs")},
                   OUT_DIR/"models"/(a["key"].replace("|","__")+".pt"))

lock={"protocol":"compact 7-feature linear classifier; validation-only selection",
      "normalization_policy": NORMALIZATION_POLICY,
      "normalization_policy_detail": "per-factory local scalers; factory_5 uses earlier feature-only unlabeled calibration",
      "seeds":SEEDS,"rounds":ROUNDS,"clients":CLIENTS,"held_out":HELD_OUT,
      "feature_names":FEATURE_NAMES,"n_parameters":len(FEATURE_NAMES)+1,
      "selection_split":"validation only","selection_metric":"SetEval row-level AP; cross-checked against sklearn average_precision_score",
      "epsilon_cap":EPSILON_CAP,"delta":DELTA,"selected_local_epochs":selected_local_epochs,
      "selected_clip_norm":selected_clip,"selected_noise_multiplier":selected_sigma,
      "selected_epsilon":privacy_epsilon(selected_sigma,steps=ROUNDS)[0],
      "selected_mean_validation_AP":float(chosen["mean_validation_AP"]),
      "candidate_summary":SWEEP_SUMMARY.to_dict("records"),"test_loaded_before_selection":False,
      "accounting_scope":"client-update mechanism only, trusted server, fixed denominator, full participation, no subsampling amplification; one final clipped client delta per round"}
(OUT_DIR/"reports"/"compact_dp_selection_lock.json").write_text(json.dumps(lock,indent=2,default=str))
pd.DataFrame(BASE_HISTORY).to_csv(OUT_DIR/"logs"/"compact_baseline_learning_curves.csv",index=False)
for seed, art in SELECTED.items():
    torch.save({"state":art["state"],"method":"DP-FedAvg-CompactLinear","seed":seed,
                "feature_names":FEATURE_NAMES,"selected_round":art["selected_round"],
                "validation_ap":art["validation_ap"],"threshold":art["threshold"],
                "local_epochs":selected_local_epochs,"clip_norm":selected_clip,
                "noise_multiplier":selected_sigma,"epsilon":art["epsilon"],"delta":DELTA},
               OUT_DIR/"models"/f"DP-FedAvg-CompactLinear__seed{seed}.pt")
print("\nSelected by validation only:", {"local_epochs":selected_local_epochs,"clip_norm":selected_clip,
       "noise_multiplier":selected_sigma,"epsilon":privacy_epsilon(selected_sigma,steps=ROUNDS)[0],
       "mean_validation_AP":float(chosen["mean_validation_AP"])})
print("Sweep wall time minutes:", round((time.time()-start)/60,2))
print("[PASS] Selection lock written before test loading.")

## 5. One final evaluation on the held-out factory

This section opens `test_site_full` only after the selected local epoch count, clipping norm, and noise multiplier have been frozen. The held-out site is transformed with its local scaler fitted on prior feature-only unlabeled calibration rows.


In [ ]:
# Test is first loaded here, after the complete configuration is frozen in the lock file.
lock_disk=json.loads((OUT_DIR/"reports"/"compact_dp_selection_lock.json").read_text())
assert lock_disk["selected_clip_norm"] == selected_clip
assert lock_disk["selected_noise_multiplier"] == selected_sigma
assert lock_disk["selected_local_epochs"] == selected_local_epochs
Xtest_all, ytest, mtest = fedkg_data.load_split("test_site_full", policy=NORMALIZATION_POLICY, feature_set="full")
Xtest = Xtest_all[:, FEATURE_IDXS].copy()
assert set(mtest["client_id"].unique()) == {HELD_OUT}
raw_failures_all = pd.read_csv(PROJECT_DIR/"data"/"raw"/"PdM_failures.csv", parse_dates=["datetime"])
FAIL_EVENTS = raw_failures_all[["machineID","datetime"]].drop_duplicates().sort_values(["machineID","datetime"])
TEST_EVAL = SetEval(mtest.reset_index(drop=True), ytest, FAIL_EVENTS)

ALL_MODELS=[]
for a in BASELINES: ALL_MODELS.append({**a,"method":a["method"]})
for a in SELECTED.values(): ALL_MODELS.append({**a,"method":"DP-FedAvg-CompactLinear"})
FINAL_ROWS=[]; FINAL_BOOT={}; FINAL_POINTS={}
for art in ALL_MODELS:
    model=make_model(len(FEATURE_NAMES),art["seed"])
    model.load_state_dict(art["state"])
    scores=predict_scores(model,Xtest)
    if not np.isfinite(scores).all(): raise AssertionError("Non-finite test predictions")
    packed=TEST_EVAL.pack(scores)
    metrics=TEST_EVAL.metrics(packed,None,art["threshold"])
    row_ap=float(average_precision_score(ytest,scores))
    if not np.isclose(float(metrics["ap"]),row_ap,atol=1e-8,rtol=1e-7):
        raise AssertionError(f"Test AP mismatch: SetEval={metrics['ap']}, sklearn={row_ap}")
    weights=draw_weights(TEST_EVAL.M,BOOTSTRAP_B,64031+art["seed"])
    reps=TEST_EVAL.bootstrap(packed,art["threshold"],weights)
    low,high=ci(reps["ap"].dropna().to_numpy(),0.95)
    FINAL_BOOT[(art["method"],art["seed"])]=reps
    FINAL_POINTS[(art["method"],art["seed"])]=metrics
    eps=privacy_epsilon(selected_sigma,steps=ROUNDS)[0] if art["method"]=="DP-FedAvg-CompactLinear" else np.nan
    FINAL_ROWS.append({"method":art["method"],"seed":art["seed"],"validation_AP":art["validation_ap"],
                       "validation_row_AP":art.get("validation_row_ap",art["validation_ap"]),
                       "selected_round":art["selected_round"],"local_epochs":art.get("local_epochs",np.nan),
                       "clip_norm":art.get("clip_norm",np.nan),
                       "noise_multiplier":selected_sigma if art["method"]=="DP-FedAvg-CompactLinear" else np.nan,
                       "epsilon_bound_training_update_channel":eps,"delta":DELTA if np.isfinite(eps) else np.nan,
                       "test_site_full_AP":metrics["ap"],"test_row_AP_sklearn":row_ap,
                       "AP_CI_low":low,"AP_CI_high":high,"ROC_AUC":metrics["roc_auc"],
                       "precision":metrics["precision"],"recall":metrics["recall"],"F1":metrics["f1"],
                       "FPR":metrics["fpr"],"event_recall":metrics["event_recall"],
                       "test_prevalence":float(np.mean(ytest)),"n_machines":TEST_EVAL.M,"n_failure_events":TEST_EVAL.n_events})
    pred=pd.DataFrame({"machineID":mtest["machineID"].to_numpy(),"datetime":mtest["datetime"].to_numpy(),
                       "score":scores.astype(np.float32),"seed":art["seed"]})
    pred.to_parquet(OUT_DIR/"predictions"/(f"{art['method']}__seed{art['seed']}_test_site_full.parquet".replace("|","__")),index=False)

FINAL=pd.DataFrame(FINAL_ROWS)
FINAL["normalization_policy"] = NORMALIZATION_POLICY
FINAL.to_csv(OUT_DIR/"results"/"compact_dp_final_test_results.csv",index=False)
SUMMARY=(FINAL.groupby("method",as_index=False).agg(seeds=("seed","nunique"),mean_test_AP=("test_site_full_AP","mean"),
          sd_test_AP=("test_site_full_AP","std"),mean_ROC_AUC=("ROC_AUC","mean"),mean_event_recall=("event_recall","mean")))
SUMMARY.to_csv(OUT_DIR/"results"/"compact_dp_final_summary.csv",index=False)
print("Selected model:", {"local_epochs":selected_local_epochs,"clip_norm":selected_clip,
      "noise_multiplier":selected_sigma,"epsilon":privacy_epsilon(selected_sigma,steps=ROUNDS)[0]})
print("\nPer-seed final results:")
print(FINAL.to_string(index=False,float_format=lambda z:f"{z:.4f}"))
print("\nSummary:")
print(SUMMARY.to_string(index=False,float_format=lambda z:f"{z:.4f}"))

## 6. Closure checks and evidence-based decision report

These checks validate the local-normalization protocol as well as reproducibility, utility, and seed stability. Passing implementation checks alone does not establish end-to-end differential privacy or generalization to new factories.


In [ ]:
checks={}
def record_check(name, ok, detail=""):
    checks[name]={"status":"PASS" if bool(ok) else "FAIL","detail":str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {detail}")

record_check("FIX-0 local normalization policy selected", NORMALIZATION_POLICY == "local",
             f"policy={NORMALIZATION_POLICY}; factory_5 uses unlabeled calibration")
record_check("FIX-1 compact model dimension is 8 parameters", len(FEATURE_NAMES)+1 == 8,
             f"features={len(FEATURE_NAMES)}; parameters={len(FEATURE_NAMES)+1}")
record_check("FIX-2 all selected DP seeds are present", set(SELECTED)==set(SEEDS), f"seeds={sorted(SELECTED)}")
record_check("FIX-3 locked noise meets epsilon cap", privacy_epsilon(selected_sigma,steps=ROUNDS)[0] <= EPSILON_CAP,
             f"sigma={selected_sigma}; epsilon={privacy_epsilon(selected_sigma,steps=ROUNDS)[0]:.4f}; cap={EPSILON_CAP}")
record_check("FIX-4 model selection used validation only",
             lock_disk["selection_split"]=="validation only" and not lock_disk["test_loaded_before_selection"])
record_check("FIX-5 final test is held-out factory only", set(mtest["client_id"].unique())=={HELD_OUT})
record_check("FIX-6 all final AP values finite", np.isfinite(FINAL["test_site_full_AP"].to_numpy()).all(), f"rows={len(FINAL)}")
record_check("FIX-7 all three seeds evaluated for selected DP setup",
             FINAL[FINAL["method"]=="DP-FedAvg-CompactLinear"]["seed"].nunique()==3)
record_check("FIX-8 validation and test AP implementations agree",
             np.allclose(FINAL["test_site_full_AP"],FINAL["test_row_AP_sklearn"],atol=1e-8,rtol=1e-7))
if any(x["status"]!="PASS" for x in checks.values()):
    raise AssertionError("Compact privacy repair implementation checks failed.")

prev=float(FINAL["test_prevalence"].iloc[0])
dp_rows=FINAL[FINAL["method"]=="DP-FedAvg-CompactLinear"].sort_values("seed")
dp_aps=dp_rows["test_site_full_AP"].to_numpy(dtype=float)
dpmean=float(np.mean(dp_aps)); dpstd=float(np.std(dp_aps,ddof=1))
cf_rows=FINAL[FINAL["method"]=="ClippedFedAvg"].sort_values("seed")
cf_aps=cf_rows["test_site_full_AP"].to_numpy(dtype=float)
cfmean=float(np.mean(cf_aps))
fa_rows=FINAL[FINAL["method"]=="FedAvg"].sort_values("seed")
famean=float(fa_rows["test_site_full_AP"].mean())
centralmean=float(FINAL.loc[FINAL["method"]=="CentralCompactLinear","test_site_full_AP"].mean())
utility_floor=3*prev
utility_pass = (dpmean >= utility_floor) and (dpmean >= 0.5*max(cfmean,1e-12))
# Conservative descriptive stability criterion: all seeds above prevalence and SD no greater than mean.
stability_pass = bool(np.all(dp_aps > prev) and dpstd <= max(dpmean,1e-12))
if utility_pass and stability_pass:
    verdict="The compact DP model meets the predeclared mean-utility and seed-stability screens. This is evidence of utility on this benchmark, not proof of parity or end-to-end DP."
elif utility_pass and not stability_pass:
    verdict="Mean-utility screen passed, but the predeclared seed-stability screen failed. Do not claim a robust successful repair yet."
elif not utility_pass and stability_pass:
    verdict="Results are comparatively stable but fail the predeclared mean-utility screen. The privacy-constrained model is not yet sufficiently useful."
else:
    verdict="The compact DP model fails the predeclared utility and/or seed-stability screens. Do not call Phase 3 privacy/utility closed."
report={"phase":"Phase 3 compact-model DP repair — local-normalization compliance v1",
        "normalization_policy": NORMALIZATION_POLICY,
        "normalization_policy_detail": "per-factory local scalers; factory_5 uses earlier feature-only unlabeled calibration",
        "inputs":{"prep":"prep_v3","main_phase3_results":str(P3_DIR/"results"/"federated_results.csv")},
        "output_directory":str(OUT_DIR),"features":FEATURE_NAMES,"n_parameters":len(FEATURE_NAMES)+1,
        "clients":CLIENTS,"held_out":HELD_OUT,"rounds":ROUNDS,"selected_local_epochs":selected_local_epochs,
        "seeds":SEEDS,"selection_protocol":"clip norm, local epochs, noise multiplier and checkpoints selected using validation AP only; test opened after lock",
        "epsilon_cap":EPSILON_CAP,"delta":DELTA,"selected_clip_norm":selected_clip,"selected_noise_multiplier":selected_sigma,
        "epsilon_bound_training_update_channel":privacy_epsilon(selected_sigma,steps=ROUNDS)[0],
        "per_seed_dp_test_AP":dp_aps.tolist(),
        "mean_seed_AP":dpmean,"sd_seed_AP":dpstd,
        "mean_utility_screen_pass":bool(utility_pass),"stability_screen_pass":bool(stability_pass),
        "means":{"selected_DP_compact_linear_test_AP":dpmean,"clipped_no_noise_compact_FedAvg_test_AP":cfmean,
                 "ordinary_compact_FedAvg_test_AP":famean,"central_compact_linear_test_AP":centralmean,
                 "test_row_prevalence":prev,"three_times_prevalence_reference":utility_floor,
                 "dp_to_clipped_AP_ratio":dpmean/max(cfmean,1e-12)},
        "decision":verdict,"checks":checks,
        "privacy_scope":"client-level add/remove bound for clipped/noised client updates only; trusted server; fixed denominator; all clients participate; no subsampling amplification; multiple local epochs are performed locally before one final clipped update per communication round",
        "limitations":["This compact model uses 7 predeclared existing features, not the full 25-feature model.",
                       "Only four simulated clients and one dataset are available.",
                       "Validation-based selection and preprocessing statistics are not covered by the reported epsilon.",
                       "The benchmark test set has been used in earlier Phase 3 analyses; this is not an untouched external replication.",
                       "The epsilon claim is scoped to the client-update mechanism, not end-to-end differential privacy.",
                       "SecureAggSim remains an arithmetic demonstration, not cryptographic secure aggregation.",
                       "Seed-stability screen is descriptive and based on three seeds; broader replications would be needed for strong general claims."]}
(OUT_DIR/"reports"/"phase3_privacy_fix_report.json").write_text(json.dumps(report,indent=2,default=str))
md_lines=["# Phase 3 — Compact DP-Federated Model with Local Normalization v1","",
          f"- Normalization policy: `{NORMALIZATION_POLICY}` (per-factory local scalers)",
          f"- Compact features: `{', '.join(FEATURE_NAMES)}` ({len(FEATURE_NAMES)} features, {len(FEATURE_NAMES)+1} parameters)",
          f"- Clients: `{', '.join(CLIENTS)}`; held-out: `{HELD_OUT}`; seeds: `{SEEDS}`; rounds: `{ROUNDS}`; selected local epochs: `{selected_local_epochs}`",
          f"- Selected clip norm: `{selected_clip}`; noise multiplier: `{selected_sigma}`",
          f"- Scoped training-update privacy bound: `epsilon={privacy_epsilon(selected_sigma,steps=ROUNDS)[0]:.4f}`, `delta={DELTA}`; study cap `epsilon <= {EPSILON_CAP}`.",
          "- Configuration and checkpoint selection used validation only; test metrics were produced after the selection lock was saved.","",
          "## Final test summary","","| Method | Seeds | Mean PR-AUC | SD PR-AUC | Mean ROC-AUC | Mean event recall |","|---|---:|---:|---:|---:|---:|"]
for _,r in SUMMARY.iterrows():
    md_lines.append(f"| {r['method']} | {int(r['seeds'])} | {r['mean_test_AP']:.4f} | {r['sd_test_AP']:.4f} | {r['mean_ROC_AUC']:.4f} | {r['mean_event_recall']:.4f} |")
md_lines += ["","## Selected DP per-seed AP",""]
for _,r in dp_rows.iterrows(): md_lines.append(f"- Seed {int(r['seed'])}: AP `{r['test_site_full_AP']:.4f}`")
md_lines += ["", "## Decision", verdict, "", "## Utility/stability screens",
              f"- Mean utility screen: **{'PASS' if utility_pass else 'FAIL'}** (mean AP={dpmean:.4f}; 3x row prevalence={utility_floor:.4f}; clipped baseline mean AP={cfmean:.4f}).",
              f"- Seed stability screen: **{'PASS' if stability_pass else 'FAIL'}** (seed AP SD={dpstd:.4f}; mean={dpmean:.4f}; APs all above row prevalence={bool(np.all(dp_aps > prev))}).",
              "", "## Implementation checks"]
for k,v in checks.items(): md_lines.append(f"- **{v['status']}** — {k}: {v['detail']}")
md_lines += ["", "## Privacy and interpretation limitations"]+[f"- {x}" for x in report["limitations"]]
md_lines += ["", "The epsilon value covers only the specified client-update mechanism under the stated threat model. It is not end-to-end differential privacy."]
(OUT_DIR/"reports"/"phase3_privacy_fix_report.md").write_text("\n".join(md_lines))
print("\n",verdict)
print("[PASS] Implementation checks complete; research success is determined separately by utility/stability screens.")
print("Report:", OUT_DIR/"reports"/"phase3_privacy_fix_report.md")
print("Results:", OUT_DIR/"results"/"compact_dp_final_test_results.csv")

## Interpretation rules

- This run is the local-normalization counterpart to the prior global-scaler experiment, not a replacement of the older output directory.
- The main privacy-utility comparison is `DP-FedAvg-CompactLinear` against `ClippedFedAvg` with the same compact architecture, features, local scaling policy, client split, and seeds.
- Do not compare results from this local-normalization run directly with previous global-normalization Phase 3 scores as if only the privacy mechanism differed; normalization is also different.
- `factory_5` scaling uses prior unlabeled calibration data, so the protocol is unsupervised site adaptation rather than strict zero-shot deployment.
- The test set has been used in earlier project experiments. This run does not provide a new untouched external replication.
- The preprocessing notebook simulates all factory partitions in one process. Per-factory scalers align the transformation protocol with the requirement, but do not by themselves prove physical raw-data isolation.
- Do not describe the accountant as end-to-end DP. `SecureAggSim` is arithmetic simulation, not cryptographic secure aggregation. Knowledge Graph and LLM requirements remain separate project components.
